In [1]:
import torch, torch.nn as nn, torch.optim as optim
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
import numpy as np, random
from PIL import Image
from tqdm import tqdm


In [2]:
class FFTMagnitude(nn.Module):
    """Compute FFT magnitude safely (float32 to avoid cuFFT half-precision crash)."""
    def forward(self, x):
        x32 = x.float()                          # avoid ComplexHalf crash
        f = torch.fft.fft2(x32)
        fshift = torch.fft.fftshift(f)
        mag = torch.abs(fshift)
        mag = torch.log1p(mag)
        return mag.to(x.dtype)


In [3]:
class Stage3Hybrid(nn.Module):
    """Stage 3 full-train hybrid based on M2TR (2023) and SFIAD (2025)."""
    def __init__(self, num_classes=2, embed_dim=1024, heads=8, depth=6):
        super().__init__()
        # --- ResNet backbone ---
        self.resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-2])  # remove avgpool+fc

        # --- Frequency branch ---
        self.fft_layer = FFTMagnitude()
        self.proj_rgb = nn.Conv2d(2048, embed_dim // 2, 1)
        self.proj_fft = nn.Conv2d(2048, embed_dim // 2, 1)

        # --- Transformer encoder ---
        enc = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=depth)

        # --- Head ---
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        rgb = self.resnet(x)              # [B,2048,10,10]
        fftmag = self.fft_layer(rgb)
        rgb = self.proj_rgb(rgb)
        fftmag = self.proj_fft(fftmag)
        fused = torch.cat([rgb, fftmag], dim=1)  # [B,1024,10,10]
        B, C, H, W = fused.shape
        tokens = fused.flatten(2).transpose(1, 2)  # [B,100,1024]
        tokens = self.transformer(tokens)
        out = tokens.mean(dim=1)
        return self.head(out)


In [4]:
def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer


In [5]:
def train_stage3():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = Stage3Hybrid().to(device)

    # ---- Simple JPEG degradation augment ----
    from io import BytesIO
    def random_jpeg(img):
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(40, 90))
        buf.seek(0)
        return Image.open(buf)

    # ---- Transforms ----
    train_tfms = transforms.Compose([
        transforms.Resize((320,320)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.1,0.05),
        transforms.RandomApply([transforms.Lambda(random_jpeg)], p=0.3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    val_tfms = transforms.Compose([
        transforms.Resize((320,320)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    # ---- Dataset paths (same as Stage 1–2B) ----
    trainset = datasets.ImageFolder(
    "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train",
        transform=train_tfms
    )
    
    valset = datasets.ImageFolder(
        "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid",
        transform=val_tfms
    )

    trainloader = DataLoader(trainset, batch_size=16, shuffle=True, num_workers=2)
    valloader   = DataLoader(valset, batch_size=16, shuffle=False, num_workers=2)

    # ---- Optimiser / loss / scaler ----
    opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler()
    epochs, best_auc = 10, 0

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        for imgs, lbls in tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}", ncols=100):
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            with torch.amp.autocast(device_type='cuda'):
                out = model(imgs)
                loss = criterion(out, lbls)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running_loss += loss.item()

      
        # ---- Validation ----
        model.eval()
        y_true, y_prob = [], []
        with torch.no_grad():
            for imgs, lbls in valloader:
                imgs = imgs.to(device)
                probs = torch.softmax(model(imgs), dim=1)[:,1].cpu().numpy()
                y_true.extend(lbls.numpy())
                y_prob.extend(probs)

        # ---- Compute metrics (with accuracy) ----
        from sklearn.metrics import accuracy_score
        y_pred = (np.array(y_prob) > 0.5).astype(int)
        auc = roc_auc_score(y_true, y_prob)
        f1 = f1_score(y_true, y_pred)
        acc = accuracy_score(y_true, y_pred)
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

        # ---- Log results ----
        print(f"Epoch {epoch+1}: loss={running_loss/len(trainloader):.4f} | "
              f"AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")

        # ---- Checkpoints ----
        torch.save(model.state_dict(), f"/kaggle/working/stage3_epoch{epoch+1}.pth")

        # track best AUROC + Accuracy
        if auc > best_auc:
            best_auc = auc
            best_acc = acc
            torch.save(model.state_dict(), "/kaggle/working/best_auc_stage3.pth")
            print(f"★ New best AUC {best_auc:.3f} (ACC={best_acc:.3f})")

    # ---- Final summary ----
    print("\n Training complete!")
    print(f" Best model achieved: AUROC = {best_auc:.3f}, Accuracy = {best_acc:.3f}")
    print(" Saved as: /kaggle/working/best_auc_stage3.pth")



In [6]:
if __name__ == "__main__":
    train_stage3()


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 173MB/s]
/tmp/ipykernel_19/1565455691.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/10: 100%|███████████████████████████████████████████████| 6250/6250 [28:07<00:00,  3.70it/s]


Epoch 1: loss=0.6995 | AUROC=0.522 | F1=0.000 | EER=0.484 | ACC=0.500
★ New best AUC 0.522 (ACC=0.500)


Epoch 2/10: 100%|███████████████████████████████████████████████| 6250/6250 [28:03<00:00,  3.71it/s]


Epoch 2: loss=0.6943 | AUROC=0.490 | F1=0.000 | EER=0.507 | ACC=0.500


Epoch 3/10: 100%|███████████████████████████████████████████████| 6250/6250 [28:00<00:00,  3.72it/s]


...

Epoch 8: loss=0.6934 | AUROC=0.499 | F1=0.000 | EER=0.501 | ACC=0.500


Epoch 9/10: 100%|███████████████████████████████████████████████| 6250/6250 [28:04<00:00,  3.71it/s]


Epoch 9: loss=0.6934 | AUROC=0.500 | F1=0.667 | EER=0.500 | ACC=0.500


Epoch 10/10: 100%|██████████████████████████████████████████████| 6250/6250 [28:02<00:00,  3.71it/s]


Epoch 10: loss=0.6934 | AUROC=0.500 | F1=0.000 | EER=0.500 | ACC=0.500

 Training complete!
 Best model achieved: AUROC = 0.522, Accuracy = 0.500
 Saved as: /kaggle/working/best_auc_stage3.pth
